In [ ]:
from pathlib import Path
import sys

# ----------------------------------------------------------------------
# 1. Make project root importable
# ----------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\91738\OneDrive\Desktop\coding\SEM 7\GAI\GAI_Project"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Project root exists: {PROJECT_ROOT.exists()}")

In [ ]:
from pathlib import Path

# The notebook is inside GAI_Project/notebooks/
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Metadata file
METADATA_FILE = RAW_DIR / "meta_Electronics.jsonl.gz"

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)
print("Metadata file:", METADATA_FILE)

In [ ]:
# Exercise 2A — Resumable metadata download

import requests
from pathlib import Path
import time

URL = (
    "https://mcauleylab.ucsd.edu/"
    "public_datasets/data/amazon_2023/raw/"
    "meta_categories/meta_Electronics.jsonl.gz"
)

CHUNK_SIZE = 2 * 1024 * 1024      # 2 MB chunks
TIMEOUT = (30, 120)               # connect timeout, read timeout
MAX_RETRIES = 5

existing_size = METADATA_FILE.stat().st_size if METADATA_FILE.exists() else 0

print(f"Starting download from byte: {existing_size:,}")
print("Connecting to dataset server...")

headers = {}
if existing_size > 0:
    headers["Range"] = f"bytes={existing_size}-"

for attempt in range(1, MAX_RETRIES + 1):

    try:
        with requests.get(
            URL,
            headers=headers,
            stream=True,
            timeout=TIMEOUT
        ) as response:

            print("HTTP status:", response.status_code)

            if response.status_code not in (200, 206):
                raise RuntimeError(
                    f"Unexpected HTTP status: {response.status_code}"
                )

            mode = "ab" if existing_size > 0 and response.status_code == 206 else "wb"

            content_range = response.headers.get("Content-Range")
            content_length = response.headers.get("Content-Length")

            print("Content-Range:", content_range)
            print("Content-Length:", content_length)
            print("Writing to:", METADATA_FILE)

            with open(METADATA_FILE, mode) as f:

                downloaded = existing_size

                for chunk in response.iter_content(
                    chunk_size=CHUNK_SIZE
                ):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)

                        print(
                            f"\rDownloaded: "
                            f"{downloaded / (1024**3):.2f} GB",
                            end=""
                        )

            print("\nDownload completed.")
            break

    except (requests.exceptions.Timeout,
            requests.exceptions.ConnectionError) as e:

        print(f"\nAttempt {attempt}/{MAX_RETRIES} failed:")
        print(type(e).__name__, e)

        if attempt < MAX_RETRIES:
            print("Will resume from current file size...")
            existing_size = METADATA_FILE.stat().st_size
            headers["Range"] = f"bytes={existing_size}-"
            time.sleep(3)
        else:
            print("Maximum retries reached.")
            print(
                f"Partial file retained at: {METADATA_FILE}"
            )

In [ ]:
# Exercise 2B — Inspect downloaded metadata

import gzip
import json
from pathlib import Path

print("Metadata file:")
print(METADATA_FILE)

print("\nFile exists:", METADATA_FILE.exists())

if METADATA_FILE.exists():
    size_gb = METADATA_FILE.stat().st_size / (1024 ** 3)
    print(f"File size: {size_gb:.2f} GB")

    print("\nReading first 5 metadata records...\n")

    with gzip.open(METADATA_FILE, "rt", encoding="utf-8") as f:
        for i in range(5):
            line = f.readline()

            if not line:
                break

            record = json.loads(line)

            print(f"Record {i + 1}:")
            print(record)
            print("-" * 80)

In [ ]:
# Exercise 2C — Category-based candidate filtering
#
# Goal:
# Identify potentially relevant headphone/earbud products
# while preserving enough metadata for later validity filtering.

import gzip
import json
import pandas as pd

# Relevant product/category terms identified during reconnaissance
CATEGORY_KEYWORDS = [
    "headphone",
    "headphones",
    "earbud",
    "earbuds",
    "earphone",
    "earphones",
]

candidate_rows = []
total_records = 0

with gzip.open(METADATA_FILE, "rt", encoding="utf-8") as f:

    for line in f:

        if not line.strip():
            continue

        record = json.loads(line)
        total_records += 1

        categories = record.get("categories") or []

        # Convert category hierarchy to searchable text
        category_text = " | ".join(
            str(category).lower()
            for category in categories
        )

        # Search category hierarchy for relevant headphone/earbud terms
        category_match = any(
            keyword in category_text
            for keyword in CATEGORY_KEYWORDS
        )

        if category_match:

            candidate_rows.append({
                "parent_asin": record.get("parent_asin"),
                "title": record.get("title"),
                "average_rating": record.get("average_rating"),
                "rating_number": record.get("rating_number"),
                "categories": categories,
                "price": record.get("price"),
                "store": record.get("store"),
            })


candidate_df = pd.DataFrame(candidate_rows)

print("Total metadata records scanned:",
      f"{total_records:,}")

print("Category candidates found:",
      f"{len(candidate_df):,}")

In [ ]:
# Verify candidate-product table

print("Candidate dataframe shape:", candidate_df.shape)
print("\nColumns:")
print(candidate_df.columns.tolist())

print("\nMissing parent ASINs:",
      candidate_df["parent_asin"].isna().sum())

print("\nDuplicate parent ASINs:",
      candidate_df["parent_asin"].duplicated().sum())

print("\nReview-count statistics:")
print(candidate_df["rating_number"].describe())

In [ ]:
# Inspect category coverage

category_counter = {}

for categories in candidate_df["categories"]:

    for category in categories:

        category_counter[category] = (
            category_counter.get(category, 0) + 1
        )

category_counts = (
    pd.Series(category_counter)
    .sort_values(ascending=False)
)

print("Most common categories among candidates:")
display(category_counts.head(30))

In [ ]:
# Inspect candidate products

display(
    candidate_df[
        [
            "parent_asin",
            "title",
            "average_rating",
            "rating_number",
            "store"
        ]
    ].head(20)
)

In [ ]:
# Save candidate-product metadata

CANDIDATE_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "candidate_products.csv"
)

CANDIDATE_FILE.parent.mkdir(parents=True, exist_ok=True)

candidate_df.to_csv(
    CANDIDATE_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved candidate products to:")
print(CANDIDATE_FILE)
print("Rows:", f"{len(candidate_df):,}")

In [ ]:
# Exercise 2D — Product validity filtering

before_count = len(candidate_df)

validity_mask = (
    candidate_df["parent_asin"].notna()
    & candidate_df["title"].notna()
    & (candidate_df["title"].astype(str).str.strip() != "")
)

valid_products_df = candidate_df.loc[
    validity_mask
].copy()

after_count = len(valid_products_df)

print("Products before validity filtering:",
      f"{before_count:,}")

print("Products after validity filtering:",
      f"{after_count:,}")

print("Removed:",
      f"{before_count - after_count:,}")

In [ ]:
print(
    "Duplicate parent ASINs:",
    valid_products_df["parent_asin"].duplicated().sum()
)

valid_products_df = (
    valid_products_df
    .drop_duplicates(subset="parent_asin")
    .reset_index(drop=True)
)

print(
    "Products after ASIN deduplication:",
    f"{len(valid_products_df):,}"
)

In [ ]:
# Exercise 2E — Review-count threshold analysis

thresholds = [25, 50, 100, 250, 500]

threshold_results = []

for threshold in thresholds:

    count = (
        valid_products_df["rating_number"]
        .fillna(0)
        .ge(threshold)
        .sum()
    )

    threshold_results.append({
        "minimum_reviews": threshold,
        "products": count
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

In [ ]:
# Save filtered candidate pool

CANDIDATE_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "candidate_products.csv"
)

CANDIDATE_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

valid_products_df.to_csv(
    CANDIDATE_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved:")
print(CANDIDATE_FILE)

print("Rows:",
      f"{len(valid_products_df):,}")

In [ ]:
# ============================================================
# Exercise 2F — Revised Semantic Product Validity Filtering
# ============================================================
#
# Goal:
# Build a high-quality pool of genuine headphone/earbud
# PRODUCTS while removing obvious accessories/components.
#
# Strategy:
#
# 1. Category evidence is the primary signal.
# 2. Strong product-identity phrases identify genuine
#    headphones/earbuds.
# 3. Accessory wording removes products that are clearly
#    accessories/components.
# 4. If a title contains BOTH strong product identity AND
#    accessory wording, the product is retained when the
#    audio product is clearly the main item.
#
# This is intentionally more conservative than the previous
# filter because false negatives are particularly harmful here.
# ============================================================

import re
import ast
import pandas as pd


# ------------------------------------------------------------
# 1. Valid audio product categories
# ------------------------------------------------------------

VALID_AUDIO_CATEGORIES = {
    "Headphones & Earbuds",
    "Earbud Headphones",
    "Over-Ear Headphones",
    "On-Ear Headphones",
    "Open-Ear Headphones",
}


# ------------------------------------------------------------
# 2. Strong audio-product identity patterns
# ------------------------------------------------------------
#
# These phrases strongly indicate that the product itself is
# a headphone / earbud / headset rather than merely an
# accessory for one.
#
# IMPORTANT:
# Do not use generic words such as "headphone", "earbud",
# "Bluetooth", etc. as standalone accessory exclusions.
# ------------------------------------------------------------

STRONG_AUDIO_PATTERNS = (
    r"\bwireless earbuds?\b|"
    r"\btrue wireless earbuds?\b|"
    r"\bbluetooth earbuds?\b|"
    r"\bwireless earphones?\b|"
    r"\bbluetooth earphones?\b|"
    r"\bwireless headphones?\b|"
    r"\bbluetooth headphones?\b|"
    r"\bnoise[- ]cancel(?:ing|ling) headphones?\b|"
    r"\bover[- ]ear headphones?\b|"
    r"\bon[- ]ear headphones?\b|"
    r"\bin[- ]ear headphones?\b|"
    r"\bin[- ]ear monitors?\b|"
    r"\bbluetooth headsets?\b|"
    r"\bgaming headsets?\b|"
    r"\bgaming headphones?\b|"
    r"\bheadphones? with mic\b|"
    r"\bearphones? with mic\b|"
    r"\bearbuds? with mic\b|"
    r"\bheadset with mic\b|"
    r"\bheadphone headset\b|"
    r"\bin[- ]ear monitor\b|"
    r"\btrue wireless\b.*\bearbuds?\b|"
    r"\bearbuds?\b.*\bwireless\b|"
    r"\bearphones?\b.*\bwireless\b|"
    r"\bheadphones?\b.*\bwireless\b"
)


# ------------------------------------------------------------
# 3. Obvious accessory/component patterns
# ------------------------------------------------------------
#
# These are terms that commonly indicate that the item itself
# is an accessory/component rather than the audio device.
# ------------------------------------------------------------

ACCESSORY_PATTERNS = (
    r"\breplacement\b|"
    r"\breplacement cable\b|"
    r"\breplacement cord\b|"
    r"\breplacement ear ?pads?\b|"
    r"\breplacement ear ?tips?\b|"
    r"\breplacement ear ?hooks?\b|"
    r"\bear ?tips?\b|"
    r"\bear ?hooks?\b|"
    r"\bear ?loops?\b|"
    r"\bearpads?\b|"
    r"\bear pads?\b|"
    r"\bheadphone pads?\b|"
    r"\bcable\b|"
    r"\bcables\b|"
    r"\bcord\b|"
    r"\bcords\b|"
    r"\bsplitter\b|"
    r"\bsplitters\b|"
    r"\badapter\b|"
    r"\badapters\b|"
    r"\badaptor\b|"
    r"\badaptors\b|"
    r"\bextension cord\b|"
    r"\bextension cable\b|"
    r"\bheadphone stand\b|"
    r"\bheadphone stands\b|"
    r"\bheadphone holder\b|"
    r"\bheadphone holders\b|"
    r"\bheadphone hanger\b|"
    r"\bheadphone hangers\b|"
    r"\bheadset stand\b|"
    r"\bheadset stands\b|"
    r"\bheadset holder\b|"
    r"\bheadset holders\b|"
    r"\bairpods? case\b|"
    r"\bairpods? cover\b|"
    r"\bairpods? skin\b|"
    r"\bprotective case\b|"
    r"\bprotective cover\b|"
    r"\bcarrying case\b|"
    r"\btravel case\b|"
    r"\bstorage case\b|"
    r"\bcase cover\b"
)


# ------------------------------------------------------------
# 4. Explicit accessory-first patterns
# ------------------------------------------------------------
#
# These are stronger than the normal accessory terms.
# They indicate that the product is clearly an accessory.
#
# Example:
# "Replacement Cable for Sennheiser HD25 Headphones"
#
# Even though "headphones" appears in the title, the actual
# product is the replacement cable.
# ------------------------------------------------------------

ACCESSORY_FIRST_PATTERNS = (
    r"^\s*replacement\b|"
    r"^\s*\d+[- ]?pack\b.*\b(?:ear ?tips?|ear ?hooks?|earpads?|"
    r"cables?|cords?|cases?|covers?)\b|"
    r"\breplacement cable\b.*\bheadphones?\b|"
    r"\breplacement cord\b.*\bheadphones?\b|"
    r"\bear ?pads?\b.*\bcompatible with\b|"
    r"\bear ?tips?\b.*\bcompatible with\b|"
    r"\bear ?hooks?\b.*\bcompatible with\b|"
    r"\bcase\b.*\bcompatible with\b.*\bairpods?\b|"
    r"\bcover\b.*\bcompatible with\b.*\bairpods?\b|"
    r"\bskin\b.*\bcompatible with\b.*\bairpods?\b|"
    r"\bprotective case\b.*\bcompatible with\b|"
    r"\bcarrying case\b.*\bheadphones?\b|"
    r"\bstorage case\b.*\bheadphones?\b"
)


# ------------------------------------------------------------
# 5. Convert categories to a Python list
# ------------------------------------------------------------

def get_category_list(value):

    if isinstance(value, (list, tuple, set)):
        return [str(x).strip() for x in value]

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)

            if isinstance(parsed, (list, tuple, set)):
                return [str(x).strip() for x in parsed]

        except (ValueError, SyntaxError):
            pass

    return [text]


# ------------------------------------------------------------
# 6. Category validation
# ------------------------------------------------------------

def has_valid_audio_category(value):

    categories = get_category_list(value)

    return any(
        category in VALID_AUDIO_CATEGORIES
        for category in categories
    )


# ------------------------------------------------------------
# 7. Strong audio-product detection
# ------------------------------------------------------------

def has_strong_audio_identity(title):

    if pd.isna(title):
        return False

    title = str(title).lower()

    return bool(
        re.search(
            STRONG_AUDIO_PATTERNS,
            title
        )
    )


# ------------------------------------------------------------
# 8. Accessory detection
# ------------------------------------------------------------

def has_accessory_wording(title):

    if pd.isna(title):
        return False

    title = str(title).lower()

    return bool(
        re.search(
            ACCESSORY_PATTERNS,
            title
        )
    )


# ------------------------------------------------------------
# 9. Accessory-first detection
# ------------------------------------------------------------

def is_accessory_first(title):

    if pd.isna(title):
        return False

    title = str(title).lower()

    return bool(
        re.search(
            ACCESSORY_FIRST_PATTERNS,
            title
        )
    )


# ------------------------------------------------------------
# 10. Apply revised semantic filtering
# ------------------------------------------------------------

before_count = len(valid_products_df)

category_mask = valid_products_df["categories"].apply(
    has_valid_audio_category
)

strong_audio_mask = valid_products_df["title"].apply(
    has_strong_audio_identity
)

accessory_mask = valid_products_df["title"].apply(
    has_accessory_wording
)

accessory_first_mask = valid_products_df["title"].apply(
    is_accessory_first
)


# ------------------------------------------------------------
# 11. Final semantic rule
# ------------------------------------------------------------
#
# A product is retained when:
#
#   A) It belongs to a valid audio category
#
# AND
#
#   B) It is NOT clearly an accessory-first product
#
# This deliberately does NOT reject every title containing
# "cable", "case", etc., because some genuine audio products
# mention included cables/cases in their titles.
#
# Strong audio identity is retained unless the title clearly
# establishes that the item being sold is an accessory.
# ------------------------------------------------------------

semantic_mask = (
    category_mask
    & ~accessory_first_mask
)


filtered_products_df = valid_products_df.loc[
    semantic_mask
].copy()


# ------------------------------------------------------------
# 12. Remove duplicate parent ASINs
# ------------------------------------------------------------

filtered_products_df = (
    filtered_products_df
    .drop_duplicates(subset="parent_asin")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 13. Results
# ------------------------------------------------------------

after_count = len(filtered_products_df)

print(
    "Products before semantic validity filtering:",
    f"{before_count:,}"
)

print(
    "Products after semantic validity filtering:",
    f"{after_count:,}"
)

print(
    "Removed:",
    f"{before_count - after_count:,}"
)

print(
    "\nProducts matching valid audio categories:",
    f"{category_mask.sum():,}"
)

print(
    "Products with strong audio-product identity:",
    f"{strong_audio_mask.sum():,}"
)

print(
    "Products containing accessory/component wording:",
    f"{accessory_mask.sum():,}"
)

print(
    "Products classified as accessory-first:",
    f"{accessory_first_mask.sum():,}"
)


# ------------------------------------------------------------
# 14. Inspect retained products
# ------------------------------------------------------------

print("\nSample of retained products:")

display(
    filtered_products_df[
        [
            "parent_asin",
            "title",
            "average_rating",
            "rating_number",
            "categories",
            "store"
        ]
    ].head(50)
)


# ------------------------------------------------------------
# 15. Inspect rejected products
# ------------------------------------------------------------

rejected_products_df = valid_products_df.loc[
    ~semantic_mask
].copy()

print(
    "\nSample of products removed by revised semantic filtering:"
)

display(
    rejected_products_df[
        [
            "parent_asin",
            "title",
            "rating_number",
            "categories"
        ]
    ].head(50)
)


# ------------------------------------------------------------
# 16. Inspect strong audio products that were retained
# ------------------------------------------------------------

retained_strong_audio = filtered_products_df[
    filtered_products_df["title"]
    .fillna("")
    .str.lower()
    .str.contains(
        STRONG_AUDIO_PATTERNS,
        regex=True
    )
].copy()

print(
    "\nStrong audio products retained:",
    f"{len(retained_strong_audio):,}"
)

display(
    retained_strong_audio[
        [
            "parent_asin",
            "title",
            "rating_number",
            "categories"
        ]
    ]
    .sort_values(
        "rating_number",
        ascending=False
    )
    .head(50)
)


# ------------------------------------------------------------
# 17. Save revised semantic-valid products
# ------------------------------------------------------------

SEMANTIC_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "semantic_valid_products.csv"
)

SEMANTIC_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

filtered_products_df.to_csv(
    SEMANTIC_FILE,
    index=False,
    encoding="utf-8"
)

print(
    "\nSaved revised semantic-valid products to:"
)

print(SEMANTIC_FILE)

print(
    "Rows:",
    f"{len(filtered_products_df):,}"
)

In [ ]:
# Exercise 2G — Review threshold comparison after validity filtering

thresholds = [25, 50, 100, 250, 500]

threshold_results = []

for threshold in thresholds:

    count = (
        filtered_products_df["rating_number"]
        .fillna(0)
        .ge(threshold)
        .sum()
    )

    threshold_results.append({
        "minimum_reviews": threshold,
        "products": count
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

In [ ]:
# ============================================================
# Exercise 2H — Apply Final Review Threshold
# ============================================================
#
# Goal:
# Create the final product pool using the selected minimum
# review threshold of 100 reviews.
#
# Final criteria:
#   1. Product passed semantic validity filtering (2F)
#   2. Product has at least 100 reviews
# ============================================================

REVIEW_THRESHOLD = 100


# ------------------------------------------------------------
# 1. Apply review threshold
# ------------------------------------------------------------

final_products_df = filtered_products_df[
    filtered_products_df["rating_number"]
    .fillna(0)
    .ge(REVIEW_THRESHOLD)
].copy()


# ------------------------------------------------------------
# 2. Remove duplicate parent ASINs
# ------------------------------------------------------------

final_products_df = (
    final_products_df
    .drop_duplicates(subset="parent_asin")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Results
# ------------------------------------------------------------

print(
    "Review threshold:",
    REVIEW_THRESHOLD
)

print(
    "Products after semantic validity filtering:",
    f"{len(filtered_products_df):,}"
)

print(
    "Final products after review threshold:",
    f"{len(final_products_df):,}"
)

print(
    "Products removed by review threshold:",
    f"{len(filtered_products_df) - len(final_products_df):,}"
)


# ------------------------------------------------------------
# 4. Basic review-count statistics
# ------------------------------------------------------------

print("\nReview-count statistics for final product pool:")

display(
    final_products_df["rating_number"]
    .describe()
    .to_frame("rating_number")
)


# ------------------------------------------------------------
# 5. Inspect final product pool
# ------------------------------------------------------------

print("\nSample of final products:")

display(
    final_products_df[
        [
            "parent_asin",
            "title",
            "average_rating",
            "rating_number",
            "categories",
            "store"
        ]
    ].head(50)
)


# ------------------------------------------------------------
# 6. Save final product pool
# ------------------------------------------------------------

FINAL_PRODUCTS_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "final_products_100_reviews.csv"
)

FINAL_PRODUCTS_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

final_products_df.to_csv(
    FINAL_PRODUCTS_FILE,
    index=False,
    encoding="utf-8"
)

print(
    "\nSaved final product pool to:"
)

print(FINAL_PRODUCTS_FILE)

print(
    "Rows:",
    f"{len(final_products_df):,}"
)

In [ ]:
# ============================================================
# Phase 4.1 — Inspect Amazon Electronics Review Dataset
# ============================================================

import gzip
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Set project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\91738\OneDrive\Desktop\coding\SEM 7\GAI\GAI_Project"
)

REVIEW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Electronics.jsonl.gz"
)

print("Review file:")
print(REVIEW_FILE)

print("\nFile exists:", REVIEW_FILE.exists())

if not REVIEW_FILE.exists():
    raise FileNotFoundError(
        f"\nCould not find the review file:\n{REVIEW_FILE}"
    )

print(
    "Compressed file size:",
    f"{REVIEW_FILE.stat().st_size / (1024**3):.2f} GB"
)


# ------------------------------------------------------------
# 2. Read 5 review records
# ------------------------------------------------------------

sample_reviews = []

with gzip.open(
    REVIEW_FILE,
    "rt",
    encoding="utf-8"
) as f:

    for _ in range(5):

        line = f.readline()

        if not line:
            break

        record = json.loads(line)

        sample_reviews.append(record)


# ------------------------------------------------------------
# 3. Display sample records
# ------------------------------------------------------------

print(
    "\nNumber of sample records:",
    len(sample_reviews)
)

for i, review in enumerate(
    sample_reviews,
    start=1
):

    print("\n" + "=" * 70)
    print(f"Review {i}")
    print("=" * 70)

    for key, value in review.items():

        print(f"{key}: {value}")


# ------------------------------------------------------------
# 4. Display available fields
# ------------------------------------------------------------

if sample_reviews:

    print("\n" + "=" * 70)
    print("Fields found in review dataset")
    print("=" * 70)

    fields = sorted(
        sample_reviews[0].keys()
    )

    for field in fields:

        print("-", field)

In [ ]:
# ============================================================
# PHASE 4 — Review Dataset Construction
# Cell 4.1 — File and Path Setup
# ============================================================
#
# Objective:
# Prepare the paths required to construct the project review
# dataset from the large Amazon Electronics review file.
#
# We will:
#   1. Locate the validated 100-product pool.
#   2. Locate the compressed Electronics review dataset.
#   3. Verify that both files exist.
#
# IMPORTANT:
# The review file is ~6 GB compressed, so it will NOT be loaded
# into memory. The actual extraction will be performed later
# using streaming line-by-line processing.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\91738\OneDrive\Desktop\coding\SEM 7\GAI\GAI_Project"
)


# ------------------------------------------------------------
# 2. Validated product pool
# ------------------------------------------------------------
#
# This is the product pool created after:
#   Phase 2 — Product Filtering
#   Phase 3 — Product Validation
#
# It should contain the validated 100-product set.
# ------------------------------------------------------------

PRODUCT_FILE = (
    PROJECT_ROOT
    / "data"
    / "validation"
    / "validated_products.csv"
)


# ------------------------------------------------------------
# 3. Amazon Electronics review file
# ------------------------------------------------------------
#
# Large compressed JSONL file containing the Amazon reviews.
# ------------------------------------------------------------

REVIEW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Electronics.jsonl.gz"
)


# ------------------------------------------------------------
# 4. Output directory for processed review data
# ------------------------------------------------------------

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 5. Verify files
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 4 — Review Dataset Construction")
print("=" * 70)

print("\nValidated product file:")
print(PRODUCT_FILE)
print("Exists:", PRODUCT_FILE.exists())

print("\nAmazon Electronics review file:")
print(REVIEW_FILE)
print("Exists:", REVIEW_FILE.exists())

print("\nProcessed-data directory:")
print(PROCESSED_DIR)
print("Exists:", PROCESSED_DIR.exists())


# ------------------------------------------------------------
# 6. Basic file-size check
# ------------------------------------------------------------

if REVIEW_FILE.exists():

    review_size_gb = REVIEW_FILE.stat().st_size / (1024 ** 3)

    print(
        f"\nCompressed review file size: "
        f"{review_size_gb:.2f} GB"
    )

In [ ]:
# ============================================================
# PHASE 4 — Review Dataset Construction
# Cell 4.2 — Locate Validated Product CSV
# ============================================================
#
# Objective:
# Find the CSV containing the manually validated 100-product
# sample from Phase 3.
#
# We are searching the project's data directory so that we
# do not accidentally use the wrong file.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Search the data directory for CSV files
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"

csv_files = sorted(DATA_DIR.rglob("*.csv"))


# ------------------------------------------------------------
# 2. Display discovered CSV files
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 4 — LOCATING VALIDATED PRODUCT CSV")
print("=" * 70)

print(
    f"\nCSV files found under:\n{DATA_DIR}\n"
)

for i, file in enumerate(csv_files, start=1):

    print(f"{i}. {file}")


print(
    f"\nTotal CSV files found: {len(csv_files)}"
)

In [ ]:
# ============================================================
# PHASE 4 — Review Dataset Construction
# Cell 4.3 — Load and Inspect Validated Product Sample
# ============================================================
#
# Objective:
# Load the manually validated 100-product sample from Phase 3.
#
# This file is the authoritative validation result and will be
# used to determine which products should have their reviews
# extracted from the large Amazon review dataset.
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Locate the validated Excel file
# ------------------------------------------------------------

VALIDATION_FILE = (
    PROJECT_ROOT
    / "data"
    / "validation"
    / "product_filter_validation.csv.xlsx"
)


# ------------------------------------------------------------
# 2. Load validation data
# ------------------------------------------------------------

validation_df = pd.read_excel(
    VALIDATION_FILE
)


# ------------------------------------------------------------
# 3. Basic inspection
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 4 — VALIDATED PRODUCT SAMPLE")
print("=" * 70)

print("\nValidation file:")
print(VALIDATION_FILE)

print("\nFile exists:", VALIDATION_FILE.exists())

print("\nShape:")
print(validation_df.shape)

print("\nColumns:")
print(validation_df.columns.tolist())


# ------------------------------------------------------------
# 4. Display validation data
# ------------------------------------------------------------

print("\nValidation sample:")

display(validation_df)


# ------------------------------------------------------------
# 5. Check missing values
# ------------------------------------------------------------

print("\nMissing values:")

display(
    validation_df.isna().sum()
)

In [ ]:
# ============================================================
# PHASE 4 — Review Dataset Construction
# Cell 4.4 — Build Validated Product ASIN Lookup
# ============================================================
#
# Objective:
# Create a set of parent ASINs corresponding to products that
# were manually marked as VALID during Phase 3.
#
# These ASINs will be used to filter the 6.03 GB Amazon review
# dataset during streaming extraction.
# ============================================================


# ------------------------------------------------------------
# 1. Inspect validation decisions
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 4 — BUILDING VALIDATED PRODUCT LOOKUP")
print("=" * 70)

print("\nValidation decisions:")

display(
    validation_df["valid"].value_counts(dropna=False)
)


# ------------------------------------------------------------
# 2. Normalize the validation column
# ------------------------------------------------------------

validation_df["valid_normalized"] = (
    validation_df["valid"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 3. Keep only manually validated products
# ------------------------------------------------------------

validated_products_df = validation_df[
    validation_df["valid_normalized"].isin(
        ["yes", "true", "1", "valid"]
    )
].copy()


# ------------------------------------------------------------
# 4. Extract parent ASINs
# ------------------------------------------------------------

validated_asins = set(
    validated_products_df["parent_asin"]
    .dropna()
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

print(
    "\nTotal manually validated products:",
    len(validated_products_df)
)

print(
    "Unique validated parent ASINs:",
    len(validated_asins)
)

print("\nFirst 20 validated parent ASINs:")

for asin in sorted(validated_asins)[:20]:
    print(asin)


# ------------------------------------------------------------
# 6. Sanity check
# ------------------------------------------------------------

if len(validated_asins) == 100:

    print(
        "\n✓ Exactly 100 validated products found."
    )

else:

    print(
        "\n⚠️ Expected 100 validated products, "
        f"but found {len(validated_asins)}."
    )

In [ ]:
# ============================================================
# PHASE 3 — Product Validation
# Step: Create Clean Validated Product Pool
# ============================================================
#
# Input:
# data/validation/product_filter_validation.csv.xlsx
#
# This is the manually validated 100-product sample.
#
# Expected:
#   100 manually inspected
#   95 valid
#   5 invalid
#
# Output:
# data/validation/validated_products.csv
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

VALIDATION_INPUT = (
    PROJECT_ROOT
    / "data"
    / "validation"
    / "product_filter_validation.csv.xlsx"
)

VALIDATED_PRODUCTS_FILE = (
    PROJECT_ROOT
    / "data"
    / "validation"
    / "validated_products.csv"
)


# ------------------------------------------------------------
# 2. Load manually validated products
# ------------------------------------------------------------

validation_df = pd.read_excel(
    VALIDATION_INPUT
)

print("=" * 70)
print("PHASE 3 — PRODUCT VALIDATION")
print("=" * 70)

print("\nValidation file:")
print(VALIDATION_INPUT)

print("File exists:", VALIDATION_INPUT.exists())

print("\nTotal validation records:", len(validation_df))


# ------------------------------------------------------------
# 3. Verify required columns
# ------------------------------------------------------------

required_columns = [
    "parent_asin",
    "title",
    "categories",
    "store",
    "average_rating",
    "rating_number",
    "valid"
]

missing_columns = [
    column
    for column in required_columns
    if column not in validation_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns verified.")


# ------------------------------------------------------------
# 4. Normalize validation labels
# ------------------------------------------------------------

validation_df["valid"] = (
    validation_df["valid"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 5. Count valid and invalid products
# ------------------------------------------------------------

valid_mask = validation_df["valid"].isin(
    ["yes", "true", "valid", "1"]
)

invalid_mask = validation_df["valid"].isin(
    ["no", "false", "invalid", "0"]
)

valid_count = valid_mask.sum()
invalid_count = invalid_mask.sum()

print("\nValidation results:")
print("Valid products:", valid_count)
print("Invalid products:", invalid_count)
print(
    "Total classified:",
    valid_count + invalid_count
)


# ------------------------------------------------------------
# 6. Check for unexpected labels
# ------------------------------------------------------------

expected_labels = {
    "yes",
    "true",
    "valid",
    "1",
    "no",
    "false",
    "invalid",
    "0"
}

unexpected_labels = sorted(
    set(validation_df["valid"].dropna().unique())
    - expected_labels
)

if unexpected_labels:
    print(
        "\nWARNING — Unexpected validation labels:"
    )
    print(unexpected_labels)
else:
    print("\nNo unexpected validation labels found.")


# ------------------------------------------------------------
# 7. Calculate acceptance rate
# ------------------------------------------------------------

total_classified = valid_count + invalid_count

if total_classified > 0:
    acceptance_rate = (
        valid_count / total_classified
    ) * 100
else:
    acceptance_rate = 0

print(
    "\nValidation acceptance rate:",
    f"{acceptance_rate:.2f}%"
)


# ------------------------------------------------------------
# 8. Extract valid products
# ------------------------------------------------------------

validated_products_df = (
    validation_df.loc[valid_mask]
    .copy()
)


# ------------------------------------------------------------
# 9. Remove duplicate parent ASINs
# ------------------------------------------------------------

before_dedup = len(validated_products_df)

validated_products_df = (
    validated_products_df
    .drop_duplicates(
        subset="parent_asin"
    )
    .reset_index(drop=True)
)

after_dedup = len(validated_products_df)

print(
    "\nValid products before deduplication:",
    before_dedup
)

print(
    "Unique validated products:",
    after_dedup
)


# ------------------------------------------------------------
# 10. Verify ASINs
# ------------------------------------------------------------

missing_asins = (
    validated_products_df["parent_asin"]
    .isna()
    .sum()
)

duplicate_asins = (
    validated_products_df["parent_asin"]
    .duplicated()
    .sum()
)

print("\nASIN validation:")
print("Missing parent ASINs:", missing_asins)
print("Duplicate parent ASINs:", duplicate_asins)


# ------------------------------------------------------------
# 11. Save clean validated product pool
# ------------------------------------------------------------

VALIDATED_PRODUCTS_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

validated_products_df.to_csv(
    VALIDATED_PRODUCTS_FILE,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 3 VALIDATION SUMMARY")
print("=" * 70)

print(
    "Products manually inspected:",
    len(validation_df)
)

print(
    "Valid products:",
    valid_count
)

print(
    "Invalid products:",
    invalid_count
)

print(
    "Validation acceptance rate:",
    f"{acceptance_rate:.2f}%"
)

print(
    "Unique validated parent ASINs:",
    len(validated_products_df)
)

print("\nSaved validated product pool to:")
print(VALIDATED_PRODUCTS_FILE)

In [ ]:
# ============================================================
# PHASE 4 — Review Dataset Construction
# Step 1: Verify Review Extraction Inputs
# ============================================================
#
# Goal:
# Confirm that all required files and the validated ASIN pool
# are available before starting the large streaming extraction.
#
# We will NOT load the 6.03 GB review file into memory.
# ============================================================

import gzip
import json
import pandas as pd


# ------------------------------------------------------------
# 1. Define Phase 4 paths
# ------------------------------------------------------------

VALIDATED_PRODUCTS_FILE = (
    PROJECT_ROOT
    / "data"
    / "validation"
    / "validated_products.csv"
)

REVIEW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Electronics.jsonl.gz"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


# ------------------------------------------------------------
# 2. Verify required files/directories
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 4 — REVIEW DATASET CONSTRUCTION")
print("Step 1 — Input Verification")
print("=" * 70)

print("\nValidated product file:")
print(VALIDATED_PRODUCTS_FILE)
print("Exists:", VALIDATED_PRODUCTS_FILE.exists())

print("\nAmazon Electronics review file:")
print(REVIEW_FILE)
print("Exists:", REVIEW_FILE.exists())

print("\nProcessed-data directory:")
print(PROCESSED_DIR)
print("Exists:", PROCESSED_DIR.exists())


# ------------------------------------------------------------
# 3. Stop if required files are missing
# ------------------------------------------------------------

if not VALIDATED_PRODUCTS_FILE.exists():
    raise FileNotFoundError(
        f"Validated product file not found:\n"
        f"{VALIDATED_PRODUCTS_FILE}"
    )

if not REVIEW_FILE.exists():
    raise FileNotFoundError(
        f"Review file not found:\n"
        f"{REVIEW_FILE}"
    )


# ------------------------------------------------------------
# 4. Create processed-data directory
# ------------------------------------------------------------

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 5. Load validated product pool
# ------------------------------------------------------------

validated_products_df = pd.read_csv(
    VALIDATED_PRODUCTS_FILE
)

print(
    "\nValidated product records:",
    len(validated_products_df)
)


# ------------------------------------------------------------
# 6. Verify parent ASIN column
# ------------------------------------------------------------

if "parent_asin" not in validated_products_df.columns:
    raise ValueError(
        "The validated product file does not contain "
        "'parent_asin'."
    )


# ------------------------------------------------------------
# 7. Build validated ASIN set
# ------------------------------------------------------------

validated_asins = set(
    validated_products_df["parent_asin"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(
    "Unique validated parent ASINs:",
    len(validated_asins)
)


# ------------------------------------------------------------
# 8. Basic ASIN sanity check
# ------------------------------------------------------------

if len(validated_asins) != 95:
    print(
        "\nWARNING:"
        f" Expected 95 validated ASINs, "
        f"but found {len(validated_asins)}."
    )
else:
    print(
        "\nValidated ASIN count confirmed: 95"
    )


# ------------------------------------------------------------
# 9. Inspect a few ASINs
# ------------------------------------------------------------

print("\nSample validated parent ASINs:")

for asin in sorted(validated_asins)[:10]:
    print(asin)


# ------------------------------------------------------------
# 10. Verify review file is readable
# ------------------------------------------------------------
#
# Read only ONE record.
# This does NOT load the 6.03 GB file.
# ------------------------------------------------------------

print("\nTesting compressed review file readability...")

with gzip.open(
    REVIEW_FILE,
    "rt",
    encoding="utf-8"
) as f:

    first_line = f.readline()

if not first_line:
    raise ValueError(
        "The review file appears to be empty."
    )

first_record = json.loads(first_line)

print("Successfully read first review record.")

print(
    "Fields in first review:",
    list(first_record.keys())
)


# ------------------------------------------------------------
# 11. Confirm important review fields
# ------------------------------------------------------------

important_review_fields = [
    "parent_asin",
    "asin",
    "rating",
    "title",
    "text",
    "timestamp",
    "verified_purchase",
    "helpful_vote"
]

available_fields = [
    field
    for field in important_review_fields
    if field in first_record
]

missing_fields = [
    field
    for field in important_review_fields
    if field not in first_record
]

print("\nImportant review fields available:")
print(available_fields)

if missing_fields:
    print(
        "\nFields not present in first record:"
    )
    print(missing_fields)
else:
    print(
        "\nAll expected review fields are present."
    )


# ------------------------------------------------------------
# 12. Final verification summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 4 INPUT VERIFICATION COMPLETE")
print("=" * 70)

print(
    "Validated products:",
    len(validated_products_df)
)

print(
    "Unique validated parent ASINs:",
    len(validated_asins)
)

print(
    "Review file available:",
    REVIEW_FILE.exists()
)

print(
    "Review file readable:",
    True
)

print(
    "Ready for streaming review extraction."
)

In [ ]:
# ============================================================
# PHASE 4 — Review Dataset Construction
# Step 2: Stream and Extract Reviews
# ============================================================
#
# Goal:
# Extract reviews belonging only to the 95 manually validated
# products.
#
# IMPORTANT:
# The 6.03 GB compressed review file is NOT loaded into memory.
# It is processed one JSON record at a time.
#
# Input:
#   data/raw/Electronics.jsonl.gz
#
# Product filter:
#   data/validation/validated_products.csv
#
# Output:
#   data/processed/validated_product_reviews.csv
#
# ============================================================

import gzip
import json
import csv
import time


# ------------------------------------------------------------
# 1. Define output path
# ------------------------------------------------------------

REVIEW_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validated_product_reviews.csv"
)


# ------------------------------------------------------------
# 2. Review fields to retain
# ------------------------------------------------------------

REVIEW_FIELDS = [
    "parent_asin",
    "asin",
    "rating",
    "title",
    "text",
    "timestamp",
    "verified_purchase",
    "helpful_vote"
]


# ------------------------------------------------------------
# 3. Prepare output file
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 4 — REVIEW DATASET CONSTRUCTION")
print("Step 2 — Streaming Review Extraction")
print("=" * 70)

print("\nInput review file:")
print(REVIEW_FILE)

print("\nOutput review file:")
print(REVIEW_OUTPUT_FILE)

print(
    "\nNumber of target products:",
    len(validated_asins)
)


# ------------------------------------------------------------
# 4. Stream through compressed JSONL
# ------------------------------------------------------------

start_time = time.time()

total_reviews = 0
matched_reviews = 0
invalid_json = 0

last_report_time = start_time

with gzip.open(
    REVIEW_FILE,
    "rt",
    encoding="utf-8"
) as input_file, open(
    REVIEW_OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as output_file:

    writer = csv.DictWriter(
        output_file,
        fieldnames=REVIEW_FIELDS,
        extrasaction="ignore"
    )

    writer.writeheader()

    for line in input_file:

        total_reviews += 1

        # ----------------------------------------------------
        # Parse JSON record
        # ----------------------------------------------------

        try:
            review = json.loads(line)

        except json.JSONDecodeError:
            invalid_json += 1
            continue


        # ----------------------------------------------------
        # Match against validated parent ASINs
        # ----------------------------------------------------

        parent_asin = review.get("parent_asin")

        if (
            parent_asin is None
            or str(parent_asin).strip()
            not in validated_asins
        ):
            continue


        # ----------------------------------------------------
        # Write matching review
        # ----------------------------------------------------

        row = {
            field: review.get(field)
            for field in REVIEW_FIELDS
        }

        writer.writerow(row)

        matched_reviews += 1


        # ----------------------------------------------------
        # Progress reporting
        # ----------------------------------------------------

        current_time = time.time()

        if current_time - last_report_time >= 30:

            elapsed = current_time - start_time

            print(
                f"Processed: {total_reviews:,} reviews | "
                f"Matched: {matched_reviews:,} | "
                f"Elapsed: {elapsed / 60:.1f} min"
            )

            last_report_time = current_time


# ------------------------------------------------------------
# 5. Final statistics
# ------------------------------------------------------------

elapsed_time = time.time() - start_time

print("\n" + "=" * 70)
print("STREAMING EXTRACTION COMPLETE")
print("=" * 70)

print(
    "Total review records processed:",
    f"{total_reviews:,}"
)

print(
    "Matching reviews extracted:",
    f"{matched_reviews:,}"
)

print(
    "Invalid JSON records:",
    f"{invalid_json:,}"
)

print(
    "Extraction time:",
    f"{elapsed_time / 60:.2f} minutes"
)

print("\nSaved review dataset to:")
print(REVIEW_OUTPUT_FILE)

In [ ]:
# ============================================================
# PHASE 4 — REVIEW DATASET CONSTRUCTION
# Step 3 — Review Dataset Quality Check
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("PHASE 4 — REVIEW DATASET CONSTRUCTION")
print("Step 3 — Review Dataset Quality Check")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate extracted review dataset
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If notebook is not being run from the project root,
# explicitly set the project root here.
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path(
        r"C:\Users\91738\OneDrive\Desktop\coding\SEM 7\GAI\GAI_Project"
    )

REVIEW_DATASET = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validated_product_reviews.csv"
)

print("\nReview dataset:")
print(REVIEW_DATASET)

print("File exists:", REVIEW_DATASET.exists())

if not REVIEW_DATASET.exists():
    raise FileNotFoundError(
        f"Review dataset not found:\n{REVIEW_DATASET}"
    )

# ------------------------------------------------------------
# 2. Load extracted review dataset
# ------------------------------------------------------------

reviews_df = pd.read_csv(REVIEW_DATASET)

print("\n" + "=" * 70)
print("BASIC DATASET INFORMATION")
print("=" * 70)

print("Total reviews:", len(reviews_df))
print("Number of columns:", len(reviews_df.columns))

print("\nColumns:")
for col in reviews_df.columns:
    print(" -", col)

# ------------------------------------------------------------
# 3. Display sample records
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE REVIEW RECORDS")
print("=" * 70)

display(reviews_df.head(5))

# ------------------------------------------------------------
# 4. Check data types
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(reviews_df.dtypes)

# ------------------------------------------------------------
# 5. Check missing values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

missing = reviews_df.isnull().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": (missing / len(reviews_df) * 100).round(2)
})

display(missing_summary)

# ------------------------------------------------------------
# 6. Check duplicate records
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE ANALYSIS")
print("=" * 70)

duplicate_rows = reviews_df.duplicated().sum()

print("Exact duplicate rows:", duplicate_rows)

# Check duplicate review IDs if available
if "user_id" in reviews_df.columns and "timestamp" in reviews_df.columns:
    duplicate_user_time = reviews_df.duplicated(
        subset=["user_id", "timestamp"]
    ).sum()

    print(
        "Duplicate user_id + timestamp combinations:",
        duplicate_user_time
    )

# ------------------------------------------------------------
# 7. Rating distribution
# ------------------------------------------------------------

if "rating" in reviews_df.columns:

    print("\n" + "=" * 70)
    print("RATING DISTRIBUTION")
    print("=" * 70)

    rating_counts = (
        reviews_df["rating"]
        .value_counts()
        .sort_index()
    )

    rating_percentages = (
        reviews_df["rating"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    rating_summary = pd.DataFrame({
        "review_count": rating_counts,
        "percentage": rating_percentages
    })

    display(rating_summary)

# ------------------------------------------------------------
# 8. Reviews per product
# ------------------------------------------------------------

if "parent_asin" in reviews_df.columns:

    print("\n" + "=" * 70)
    print("REVIEWS PER PRODUCT")
    print("=" * 70)

    reviews_per_product = (
        reviews_df
        .groupby("parent_asin")
        .size()
        .sort_values(ascending=False)
    )

    print("Unique products represented:",
          reviews_per_product.index.nunique())

    print("\nReview-count statistics:")

    display(
        reviews_per_product.describe()
    )

# ------------------------------------------------------------
# 9. Products with zero extracted reviews
# ------------------------------------------------------------

validated_products_file = (
    PROJECT_ROOT
    / "data"
    / "validation"
    / "validated_products.csv"
)

if validated_products_file.exists():

    validated_products = pd.read_csv(
        validated_products_file
    )

    validated_asins = set(
        validated_products["parent_asin"]
        .dropna()
        .astype(str)
    )

    extracted_asins = set(
        reviews_df["parent_asin"]
        .dropna()
        .astype(str)
    )

    products_without_reviews = (
        validated_asins - extracted_asins
    )

    print("\n" + "=" * 70)
    print("PRODUCT COVERAGE")
    print("=" * 70)

    print(
        "Validated products:",
        len(validated_asins)
    )

    print(
        "Products with extracted reviews:",
        len(extracted_asins)
    )

    print(
        "Validated products with no extracted reviews:",
        len(products_without_reviews)
    )

    if products_without_reviews:
        print("\nProducts without extracted reviews:")
        for asin in sorted(products_without_reviews):
            print(" -", asin)

# ------------------------------------------------------------
# 10. Review text availability
# ------------------------------------------------------------

if "text" in reviews_df.columns:

    empty_text = (
        reviews_df["text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print("\n" + "=" * 70)
    print("REVIEW TEXT QUALITY")
    print("=" * 70)

    print("Empty review texts:", empty_text)

    print(
        "Non-empty review texts:",
        len(reviews_df) - empty_text
    )

# ------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 4 — STEP 3 SUMMARY")
print("=" * 70)

print("Total extracted reviews:", len(reviews_df))
print("Unique products:", reviews_df["parent_asin"].nunique())

if "rating" in reviews_df.columns:
    print(
        "Average review rating:",
        round(reviews_df["rating"].mean(), 3)
    )

print("Exact duplicate rows:", duplicate_rows)

print("\nReview dataset quality check completed.")

In [ ]:
# ======================================================================
# PHASE 4 — REVIEW DATASET CONSTRUCTION
# Step 4 — Review Cleaning
# Path Configuration
# ======================================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("PHASE 4 — REVIEW DATASET CONSTRUCTION")
print("Step 4 — Review Cleaning")
print("=" * 70)

# Notebook is inside:
# GAI_Project/notebooks/
#
# Therefore, move one directory upward to reach:
# GAI_Project/

PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validated_product_reviews.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cleaned_product_reviews.csv"
)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Input file:   {INPUT_FILE}")
print(f"Output file:  {OUTPUT_FILE}")

# Verify input
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"\nInput review dataset not found:\n{INPUT_FILE}"
    )

print("\nInput review dataset found successfully.")
# ----------------------------------------------------------------------
# 2. Load extracted review dataset
# ----------------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("\n" + "=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------------------------------------------------
# 3. Preserve original row count
# ----------------------------------------------------------------------

original_rows = len(df)

# ----------------------------------------------------------------------
# 4. Remove exact duplicate rows
# ----------------------------------------------------------------------

duplicate_count = df.duplicated().sum()

df = df.drop_duplicates().copy()

print("\n" + "=" * 70)
print("DUPLICATE REMOVAL")
print("=" * 70)

print(f"Exact duplicate rows removed: {duplicate_count:,}")
print(f"Rows remaining: {len(df):,}")

# ----------------------------------------------------------------------
# 5. Remove reviews with missing/empty review text
# ----------------------------------------------------------------------

# Convert text to string only after identifying missing values.
# This prevents NaN values from becoming the literal string "nan".

missing_text_count = df["text"].isna().sum()

# Treat whitespace-only text as empty.
empty_text_mask = (
    df["text"].fillna("").astype(str).str.strip().eq("")
)

empty_text_count = empty_text_mask.sum()

df = df.loc[~empty_text_mask].copy()

print("\n" + "=" * 70)
print("REVIEW TEXT CLEANING")
print("=" * 70)

print(f"Missing text values: {missing_text_count:,}")
print(f"Empty/whitespace-only texts: {empty_text_count:,}")
print(f"Rows remaining: {len(df):,}")

# ----------------------------------------------------------------------
# 6. Clean review title and text formatting
# ----------------------------------------------------------------------

df["title"] = (
    df["title"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ----------------------------------------------------------------------
# 7. Create combined review field
# ----------------------------------------------------------------------

df["review_text"] = (
    df["title"].str.strip()
    + ". "
    + df["text"].str.strip()
).str.strip()

# Remove a trailing period if the review consisted only of a title.
df["review_text"] = (
    df["review_text"]
    .str.replace(r"\.\s*\.", ".", regex=True)
    .str.strip()
)

# ----------------------------------------------------------------------
# 8. Remove unusably short reviews
# ----------------------------------------------------------------------

# We retain short but meaningful reviews such as:
# "Great!" or "Terrible."
#
# Therefore, we use a conservative character threshold rather than
# aggressively removing short customer opinions.

MIN_REVIEW_LENGTH = 5

short_review_mask = (
    df["review_text"].str.len() < MIN_REVIEW_LENGTH
)

short_review_count = short_review_mask.sum()

df = df.loc[~short_review_mask].copy()

print("\n" + "=" * 70)
print("SHORT REVIEW FILTER")
print("=" * 70)

print(f"Minimum review length: {MIN_REVIEW_LENGTH} characters")
print(f"Very short reviews removed: {short_review_count:,}")
print(f"Rows remaining: {len(df):,}")

# ----------------------------------------------------------------------
# 9. Check duplicate review text
# ----------------------------------------------------------------------

duplicate_review_text_count = df["review_text"].duplicated().sum()

print("\n" + "=" * 70)
print("DUPLICATE REVIEW-TEXT CHECK")
print("=" * 70)

print(
    f"Duplicate review_text values remaining: "
    f"{duplicate_review_text_count:,}"
)

# IMPORTANT:
# We do NOT remove these automatically.
#
# Different customers can legitimately submit identical short reviews.
# We only removed exact duplicate records above.

# ----------------------------------------------------------------------
# 10. Validate ratings
# ----------------------------------------------------------------------

invalid_rating_mask = ~df["rating"].isin([1, 2, 3, 4, 5])

invalid_rating_count = invalid_rating_mask.sum()

print("\n" + "=" * 70)
print("RATING VALIDATION")
print("=" * 70)

print(f"Invalid rating records: {invalid_rating_count:,}")

if invalid_rating_count > 0:
    print("WARNING: Invalid ratings detected.")
else:
    print("All remaining ratings are valid (1–5).")

# ----------------------------------------------------------------------
# 11. Final dataset information
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL CLEANED DATASET")
print("=" * 70)

print(f"Original rows:        {original_rows:,}")
print(f"Final cleaned rows:   {len(df):,}")
print(f"Rows removed:         {original_rows - len(df):,}")

print(f"\nUnique products:       {df['parent_asin'].nunique():,}")
print(f"Average rating:        {df['rating'].mean():.2f}")

print("\nColumns:")
for column in df.columns:
    print(f"  - {column}")

# ----------------------------------------------------------------------
# 12. Save cleaned dataset
# ----------------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("CLEANING COMPLETE")
print("=" * 70)

print(f"Saved cleaned dataset to:")
print(OUTPUT_FILE)

print(f"\nFinal rows: {len(df):,}")
print(f"Final columns: {len(df.columns)}")

In [ ]:
# ======================================================================
# PHASE 4 — REVIEW DATASET CONSTRUCTION
# Step 5 — Dataset Profiling / Exploratory Analysis
# ======================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("PHASE 4 — REVIEW DATASET CONSTRUCTION")
print("Step 5 — Dataset Profiling / Exploratory Analysis")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. Load cleaned review dataset
# ----------------------------------------------------------------------

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cleaned_product_reviews.csv"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned review dataset not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print(f"\nDataset loaded successfully.")
print(f"Total reviews: {len(df):,}")
print(f"Unique products: {df['parent_asin'].nunique():,}")

# ----------------------------------------------------------------------
# 2. Reviews per product
# ----------------------------------------------------------------------

reviews_per_product = (
    df.groupby("parent_asin")
    .size()
    .sort_values(ascending=False)
)

print("\n" + "=" * 70)
print("REVIEWS PER PRODUCT")
print("=" * 70)

print(f"Products: {len(reviews_per_product):,}")
print(f"Mean:     {reviews_per_product.mean():.2f}")
print(f"Median:   {reviews_per_product.median():.2f}")
print(f"Std:      {reviews_per_product.std():.2f}")
print(f"Minimum:  {reviews_per_product.min():,}")
print(f"Maximum:  {reviews_per_product.max():,}")

print("\nPercentiles:")
print(
    reviews_per_product
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95])
    .to_string()
)

# ----------------------------------------------------------------------
# 3. Rating distribution
# ----------------------------------------------------------------------

rating_distribution = (
    df["rating"]
    .value_counts()
    .sort_index()
)

rating_percentage = (
    df["rating"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

rating_summary = pd.DataFrame({
    "review_count": rating_distribution,
    "percentage": rating_percentage.round(2)
})

print("\n" + "=" * 70)
print("RATING DISTRIBUTION")
print("=" * 70)

print(rating_summary)

# ----------------------------------------------------------------------
# 4. Verified purchase distribution
# ----------------------------------------------------------------------

verified_distribution = (
    df["verified_purchase"]
    .value_counts()
)

verified_percentage = (
    df["verified_purchase"]
    .value_counts(normalize=True)
    * 100
)

verified_summary = pd.DataFrame({
    "review_count": verified_distribution,
    "percentage": verified_percentage.round(2)
})

print("\n" + "=" * 70)
print("VERIFIED PURCHASE DISTRIBUTION")
print("=" * 70)

print(verified_summary)

# ----------------------------------------------------------------------
# 5. Helpful vote distribution
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("HELPFUL VOTE DISTRIBUTION")
print("=" * 70)

print(df["helpful_vote"].describe())

print("\nHelpful-vote percentiles:")
print(
    df["helpful_vote"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_string()
)

# ----------------------------------------------------------------------
# 6. Review length analysis
# ----------------------------------------------------------------------

df["review_length_chars"] = (
    df["review_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

df["review_length_words"] = (
    df["review_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

print("\n" + "=" * 70)
print("REVIEW LENGTH DISTRIBUTION")
print("=" * 70)

print("\nCharacters:")
print(df["review_length_chars"].describe())

print("\nWords:")
print(df["review_length_words"].describe())

print("\nWord-count percentiles:")
print(
    df["review_length_words"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_string()
)

# ----------------------------------------------------------------------
# 7. Product-level representation
# ----------------------------------------------------------------------

product_distribution = pd.DataFrame({
    "review_count": reviews_per_product
})

print("\n" + "=" * 70)
print("PRODUCT REPRESENTATION")
print("=" * 70)

print("\nProducts by review-count ranges:")

bins = [0, 10, 25, 50, 100, 250, 500, 1000, np.inf]

labels = [
    "1–10",
    "11–25",
    "26–50",
    "51–100",
    "101–250",
    "251–500",
    "501–1000",
    "1000+"
]

product_distribution["review_range"] = pd.cut(
    product_distribution["review_count"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

range_summary = (
    product_distribution["review_range"]
    .value_counts()
    .sort_index()
)

print(range_summary)

# ----------------------------------------------------------------------
# 8. Top products by number of extracted reviews
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 20 PRODUCTS BY REVIEW COUNT")
print("=" * 70)

top_products = (
    df.groupby("parent_asin")
    .agg(
        review_count=("parent_asin", "size"),
        average_rating=("rating", "mean")
    )
    .sort_values("review_count", ascending=False)
    .head(20)
)

top_products["average_rating"] = (
    top_products["average_rating"].round(2)
)

print(top_products)

# ----------------------------------------------------------------------
# 9. Overall dataset statistics
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL DATASET SUMMARY")
print("=" * 70)

print(f"Total reviews:              {len(df):,}")
print(f"Unique products:             {df['parent_asin'].nunique():,}")
print(f"Average rating:              {df['rating'].mean():.2f}")
print(f"Median rating:               {df['rating'].median():.1f}")
print(f"Average review length:       {df['review_length_words'].mean():.1f} words")
print(f"Median review length:        {df['review_length_words'].median():.1f} words")
print(f"Average helpful votes:       {df['helpful_vote'].mean():.2f}")
print(f"Median helpful votes:        {df['helpful_vote'].median():.1f}")
print(
    f"Verified purchase rate:      "
    f"{df['verified_purchase'].mean() * 100:.2f}%"
)

# ----------------------------------------------------------------------
# 10. Final profiling status
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 4 — STEP 5 COMPLETE")
print("=" * 70)

print(
    "Dataset profiling completed successfully."
)

print(
    "\nNo records were removed or modified during this profiling step."
)

In [ ]:
# ======================================================================
# PHASE 4 — REVIEW DATASET CONSTRUCTION
# Step 6 — Product-Level Review Sampling Analysis
# ======================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("PHASE 4 — REVIEW DATASET CONSTRUCTION")
print("Step 6 — Product-Level Review Sampling Analysis")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. Define project paths
# ----------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

# If notebook is inside the notebooks/ directory, move to project root
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cleaned_product_reviews.csv"
)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Input file:  {INPUT_FILE}")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned review dataset not found:\n{INPUT_FILE}"
    )

# ----------------------------------------------------------------------
# 2. Load cleaned review dataset
# ----------------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(f"Total reviews: {len(df):,}")
print(f"Unique products: {df['parent_asin'].nunique():,}")

# ----------------------------------------------------------------------
# 3. Calculate reviews per product
# ----------------------------------------------------------------------

reviews_per_product = (
    df.groupby("parent_asin")
      .size()
      .sort_values(ascending=False)
)

print("\n" + "=" * 70)
print("REVIEWS PER PRODUCT")
print("=" * 70)

print(f"Number of products: {len(reviews_per_product):,}")
print(f"Minimum reviews/product: {reviews_per_product.min():,}")
print(f"Median reviews/product:  {reviews_per_product.median():,.0f}")
print(f"Mean reviews/product:    {reviews_per_product.mean():,.2f}")
print(f"Maximum reviews/product: {reviews_per_product.max():,}")

# ----------------------------------------------------------------------
# 4. Evaluate different per-product sampling caps
# ----------------------------------------------------------------------

CAPS = [25, 50, 100, 150, 200]

sampling_results = []

for cap in CAPS:

    reviews_after_cap = reviews_per_product.clip(upper=cap)

    sampling_results.append({
        "max_reviews_per_product": cap,
        "products_included": len(reviews_after_cap),
        "total_reviews": int(reviews_after_cap.sum()),
        "reviews_removed": int(len(df) - reviews_after_cap.sum()),
        "percentage_retained": round(
            (reviews_after_cap.sum() / len(df)) * 100, 2
        )
    })

sampling_df = pd.DataFrame(sampling_results)

print("\n" + "=" * 70)
print("SAMPLING SCENARIO ANALYSIS")
print("=" * 70)

print(sampling_df.to_string(index=False))

# ----------------------------------------------------------------------
# 5. Show how many products fall below each cap
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("PRODUCT COVERAGE UNDER EACH CAP")
print("=" * 70)

for cap in CAPS:

    products_at_or_below = (reviews_per_product <= cap).sum()
    products_above = (reviews_per_product > cap).sum()

    print(
        f"\nCap = {cap} reviews/product"
        f"\n  Products with <= {cap} reviews: {products_at_or_below}"
        f"\n  Products with >  {cap} reviews: {products_above}"
    )

# ----------------------------------------------------------------------
# 6. Identify products that would be heavily downsampled
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP PRODUCTS AFFECTED BY SAMPLING")
print("=" * 70)

top_products = reviews_per_product.head(20).reset_index()
top_products.columns = ["parent_asin", "original_review_count"]

top_products["after_25_cap"] = top_products[
    "original_review_count"
].clip(upper=25)

top_products["after_50_cap"] = top_products[
    "original_review_count"
].clip(upper=50)

top_products["after_100_cap"] = top_products[
    "original_review_count"
].clip(upper=100)

print(top_products.to_string(index=False))

# ----------------------------------------------------------------------
# 7. Final summary
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 4 — STEP 6 SUMMARY")
print("=" * 70)

print(f"Current cleaned dataset: {len(df):,} reviews")
print(f"Validated products: {df['parent_asin'].nunique():,}")

print("\nSampling scenarios evaluated:")
for cap in CAPS:
    total = int(reviews_per_product.clip(upper=cap).sum())
    print(f"  - {cap:>3} reviews/product → {total:,} reviews")

print("\nNo records were removed or modified during this step.")
print("This step only evaluates possible sampling strategies.")
print("=" * 70)

In [ ]:
# ======================================================================
# PHASE 4 — REVIEW DATASET CONSTRUCTION
# Step 7 — Create Final Product-Level Sample
# ======================================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("PHASE 4 — REVIEW DATASET CONSTRUCTION")
print("Step 7 — Create Final Product-Level Sample")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. Project paths
# ----------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cleaned_product_reviews.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "sampled_product_reviews_100.csv"
)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Input file:  {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")

# ----------------------------------------------------------------------
# 2. Load cleaned review dataset
# ----------------------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned review dataset not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print("\n" + "=" * 70)
print("INPUT DATASET")
print("=" * 70)

print(f"Total reviews: {len(df):,}")
print(f"Unique products: {df['parent_asin'].nunique():,}")

# ----------------------------------------------------------------------
# 3. Validate required columns
# ----------------------------------------------------------------------

required_columns = [
    "parent_asin",
    "timestamp"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ----------------------------------------------------------------------
# 4. Deterministic product-level sampling
#
# Maximum 100 reviews per product.
#
# Reviews are sorted chronologically and the first 100 reviews
# for each product are retained.
#
# Using group_keys=False keeps parent_asin as a normal column.
# ----------------------------------------------------------------------

MAX_REVIEWS_PER_PRODUCT = 100

sampled_df = (
    df.sort_values(
        by=["parent_asin", "timestamp"],
        ascending=[True, True]
    )
    .groupby(
        "parent_asin",
        group_keys=False
    )
    .head(MAX_REVIEWS_PER_PRODUCT)
    .copy()
)

# ----------------------------------------------------------------------
# 5. Sort final dataset for reproducibility/readability
# ----------------------------------------------------------------------

sampled_df = sampled_df.sort_values(
    by=["parent_asin", "timestamp"],
    ascending=[True, True]
).reset_index(drop=True)

# ----------------------------------------------------------------------
# 6. Validate sampling result
# ----------------------------------------------------------------------

reviews_per_product = sampled_df.groupby(
    "parent_asin"
).size()

print("\n" + "=" * 70)
print("SAMPLING RESULTS")
print("=" * 70)

print(f"Original reviews:       {len(df):,}")
print(f"Sampled reviews:        {len(sampled_df):,}")
print(f"Reviews removed:        {len(df) - len(sampled_df):,}")
print(f"Unique products:        {sampled_df['parent_asin'].nunique():,}")
print(f"Maximum reviews/product:{reviews_per_product.max():,}")
print(f"Minimum reviews/product:{reviews_per_product.min():,}")

# Confirm no product exceeds the 100-review cap
if reviews_per_product.max() > MAX_REVIEWS_PER_PRODUCT:
    raise ValueError(
        "Sampling validation failed: a product contains more than "
        f"{MAX_REVIEWS_PER_PRODUCT} reviews."
    )

print("\n100-review-per-product cap verified successfully.")

# ----------------------------------------------------------------------
# 7. Display distribution
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("REVIEWS PER PRODUCT AFTER SAMPLING")
print("=" * 70)

print(
    reviews_per_product.describe()
)

# ----------------------------------------------------------------------
# 8. Save final sampled dataset
# ----------------------------------------------------------------------

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

sampled_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("FINAL SAMPLE CREATED")
print("=" * 70)

print(f"Saved to:")
print(OUTPUT_FILE)

print(f"\nFinal rows: {len(sampled_df):,}")
print(f"Final columns: {len(sampled_df.columns)}")

print("\nColumns:")
for column in sampled_df.columns:
    print(f"  - {column}")

print("\n" + "=" * 70)
print("PHASE 4 — STEP 7 COMPLETE")
print("=" * 70)